# Exemplo: Metadados
-------------------

Este exemplo mostra como adicionar metadados como `groups` e `sample_weight` ao experionml.

Importe o conjunto de dados de vinhos de [sklearn.datasets](https://scikit-learn.org/stable/datasets/index.html#breast-cancer-wisconsin-diagnostic-dataset). Este é um conjunto de dados pequeno e fácil de treinar cujo objetivo é classificar os vinhos em três grupos, isto é, identificar de qual cultivador eles vêm, usando variáveis baseadas nos resultados de análises químicas.

## Carregar os dados

In [1]:
# Import packages
import numpy as np
from sklearn.datasets import load_wine
from experionml import ExperionMLClassifier

In [2]:
# Carregue os dados
X, y = load_wine(return_X_y=True, as_frame=True)

# Let's have a look
X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [3]:
# Crie grupos e sample_weights fictícios para as linhas
groups = np.random.randint(5, size=X.shape[0])
sample_weight = np.random.randint(5, size=X.shape[0])
print(groups)

[4 4 2 3 2 1 0 1 4 1 2 0 0 2 3 1 2 1 2 0 2 0 4 3 3 4 2 2 1 3 3 0 4 0 4 4 3
 1 4 2 0 4 2 1 1 3 0 0 2 4 0 4 4 1 0 0 4 4 4 2 1 3 4 2 0 3 4 0 4 2 1 2 1 3
 2 1 0 1 1 0 0 4 4 0 1 1 1 4 1 3 1 2 1 0 1 0 1 3 1 2 2 3 1 2 2 3 2 1 2 0 4
 1 2 2 4 4 0 3 0 3 0 1 1 0 2 0 3 4 1 2 2 1 4 1 4 3 1 3 3 1 3 4 3 2 3 1 0 3
 0 0 1 4 0 3 4 4 2 1 0 1 0 1 1 4 1 1 4 4 0 1 0 3 2 2 1 1 0 0]


## Executar o pipeline

Adicione os metadados ao construtor. Mantemos `index=True` para demonstrar que a funcionalidade de grupos funciona.  
Quando grupos são especificados, `test_size` determina o número de grupos no conjunto de teste.

In [4]:
experionml = ExperionMLClassifier(
    X,
    y=y,
    index=True,
    metadata={"groups": groups, "sample_weight": sample_weight},
    test_size=1,
    verbose=2,
    random_state=1,
)

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Multiclass classification.

Estatísticas do conjunto de dados ==================== >>
Formato: (178, 14)
Tamanho do conjunto de train: 145
Tamanho do conjunto de test: 33
-------------------------------------
Memória: 25.53 kB
Escalonado: False
Valores atípicos: 10 (0.5%)



In [5]:
# Mostre que todas as linhas no conjunto de teste pertencem ao mesmo grupo
experionml.metadata["groups"].loc[experionml.test.index]

16     2
124    2
99     2
26     2
2      2
108    2
112    2
172    2
59     2
143    2
74     2
106    2
173    2
130    2
63     2
4      2
13     2
20     2
39     2
113    2
69     2
42     2
91     2
27     2
104    2
129    2
48     2
103    2
71     2
18     2
156    2
100    2
10     2
Name: groups, dtype: int64

In [6]:
# Visualize os grupos
experionml.plot_data_splits()

In [7]:
experionml.scale()

Ajustando Scaler...
Escalonando as variáveis...


In [8]:
# Observe que os sample weights são passados ao escalonador
experionml.pipeline[0].get_metadata_routing()

{'fit': {'sample_weight': True}}

In [9]:
experionml.run("LR")


Training ========================= >>
Models: LR
Metric: f1_weighted


Results for LogisticRegression:
Fit ---------------------------------------------
Train evaluation --> f1_weighted: 1.0
Test evaluation --> f1_weighted: 1.0
Time elapsed: 0.041s
-------------------------------------------------
Time: 0.041s


Resultados finais ==================== >>
Tempo total: 0.047s
-------------------------------------
LogisticRegression --> f1_weighted: 1.0


In [10]:
# O mesmo vale para os modelos...
experionml.lr.estimator.get_metadata_routing()

{'fit': {'sample_weight': True}, 'score': {'sample_weight': None}}

In [11]:
# ... and metrics
experionml._metric[0].get_metadata_routing()

{'score': {'sample_weight': True}}

In [12]:
experionml.lr.cross_validate()

Applying cross-validation...


,train_f1_weighted,test_f1_weighted,time
0,1.000000,0.985566,0.019094
1,1.000000,0.988241,0.026340
2,1.000000,1.000000,0.031221
3,1.000000,0.951256,0.028923
4,1.000000,1.000000,0.032375
mean,1.000000,0.985012,0.027591


In [13]:
experionml.plot_cv_splits()